# Session 1: S, I and R - three boxes with arrows
**A. Real-world problem.** When a new infection appears in a town, why do some outbreaks
explode while others fizzle out? Modellers sort people into three boxes:
**S**usceptible (could catch it), **I**nfected (have it and can pass it on) and
**R**ecovered (had it, now immune). Two arrows move people: S -> I (infection) and
I -> R (recovery).

**Plan.** (1) move people by hand for one day, (2) let the computer repeat that for many
small time steps, (3) find the day with the most people infected, (4) see that everything
depends on how many people each case infects.

**Important.** This is a teaching toy with invented numbers. It is not a forecast, not
medical advice, and not a basis for public-health decisions.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: one small step
A town of N = 1000 people, one of them infected. In a tiny time step `dt` (in days):
* new infections = beta * S * I / N * dt  (beta = contacts that pass the infection, per day)
* new recoveries = gamma * I * dt          (gamma = 1 / days a person stays infectious)

In [ ]:
def one_step(S, I, R, beta, gamma, N, dt):
    """Move people between the boxes for a time dt (days). Returns the new (S, I, R)."""
    new_infections = beta * S * I / N * dt
    new_recoveries = gamma * I * dt
    return S - new_infections, I + new_infections - new_recoveries, R + new_recoveries

## Step 2: repeat many times
We use steps of 0.1 day and keep one value per day.

In [ ]:
def run_outbreak(beta, gamma, N=1000, I0=1, days=120, dt=0.1):
    """Return arrays S, I, R with one value per day (length days + 1)."""
    S, I, R = float(N - I0), float(I0), 0.0
    out = [(S, I, R)]
    per_day = int(round(1 / dt))
    for _ in range(days):
        for _ in range(per_day):
            S, I, R = one_step(S, I, R, beta, gamma, N, dt)
        out.append((S, I, R))
    S_, I_, R_ = np.array(out).T
    return S_, I_, R_

## Step 3: how many does each case infect?
One infected person stays infectious for 1/gamma days and meets people at rate beta, so
infects about **beta / gamma** others (when almost everybody is still susceptible). We call
this number R0, the *basic reproduction number*. If R0 is above 1, each case replaces itself
with more than one case and the outbreak grows. If it is below 1, it dies away.

In [ ]:
def r0(beta, gamma):
    return beta / gamma


def generations(R0, n=6):
    """Cases in generation 0, 1, 2, ... if every case infects R0 others (early phase)."""
    return [R0 ** k for k in range(n)]


def peak_of(I):
    """(day of the peak, number infected at the peak)."""
    k = int(np.argmax(I))
    return k, float(I[k])

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    beta, gamma = 0.5, 0.25
    S, I, R = run_outbreak(beta, gamma)
    day, top = peak_of(I)
    print("R0 =", r0(beta, gamma))
    print("people at the end: S, I, R =", round(S[-1], 1), round(I[-1], 2), round(R[-1], 1))
    print("peak day and peak infected:", day, round(top, 1))
    print("generations for R0 = 2:", generations(2.0))
    print("generations for R0 = 0.5:", generations(0.5))

    d = np.arange(len(S))
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.plot(d, S, label="S susceptible")
    ax.plot(d, I, label="I infected")
    ax.plot(d, R, label="R recovered")
    ax.axvline(day, color="gray", ls=":")
    ax.set_xlabel("day")
    ax.set_ylabel("people (out of 1000)")
    ax.set_title("A toy outbreak (R0 = 2): infected people rise, peak, fall")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "sir_curves.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.5, 4))
    for b, label in ((0.5, "R0 = 2.0 (grows)"), (0.25, "R0 = 1.0 (borderline)"), (0.125, "R0 = 0.5 (dies out)")):
        _, Ib, _ = run_outbreak(b, gamma)
        ax2.plot(d, Ib, label=label)
    ax2.set_xlabel("day")
    ax2.set_ylabel("people infected now")
    ax2.set_title("The curve depends on R0 = beta / gamma")
    ax2.legend()
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "r0_compare.png", dpi=120)

## Where this stops
Real people differ in age and behaviour, meet in groups, and are counted late. Session 2
turns these boxes and arrows into equations; Session 3 fits them to noisy data.

## Coding tasks
1. **Easy.** Change `beta` and `gamma` in `lesson.py`: find the smallest beta (in steps of 0.05, gamma = 0.25) for which the highest number of infected people is above 10.
2. **Stretch.** Write a function `final_recovered(beta, gamma)` that returns R on day 120 and print it for R0 = 0.5, 1 and 2. Which is surprising?
3. **Explain.** Explain in your own words why the infected curve must turn over, using the words susceptible and recovered.

## Assessment
- **A. Conceptual.** Why does the number of infected people fall after the peak even if nobody changes their behaviour?
- **B. Numerical.** A town has beta = 0.3 and gamma = 0.1 per day. Find R0 and the number of cases in generation 3, starting from one case.
- **C. Graph and simulation interpretation.** In the S, I, R figure, on which day is the peak, roughly how many people are infected then, and what happens to R at that moment?
- **D. Design and optimisation challenge.** Design: with gamma = 0.25 per day, the town's contact rate is beta = 0.5. A school-closure plan could reduce beta. What beta makes R0 equal to exactly 1, and what happens if the plan reaches beta = 0.2? State one reason your answer might be too optimistic.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    # hand calculation of one step of one day (beta 0.5, gamma 0.25, N 1000, S 999, I 1)
    S, I, R = L.one_step(999.0, 1.0, 0.0, 0.5, 0.25, 1000, 1.0)
    checks.append(("worked example: one day moves 0.4995 people S->I and 0.25 people I->R",
                   abs(S - 998.5005) < 1e-9 and abs(I - 1.2495) < 1e-9 and abs(R - 0.25) < 1e-9))
    checks.append(("one step keeps the total of people", abs(S + I + R - 1000) < 1e-9))

    S, I, R = L.run_outbreak(0.5, 0.25)
    checks.append(("S + I + R stays 1000 on every day", float(np.abs(S + I + R - 1000).max()) < 1e-6))
    checks.append(("nobody is negative", bool(min(S.min(), I.min(), R.min()) > -1e-9)))
    checks.append(("day 10 (R0 = 2): S, I, R are about 978.0, 11.4, 10.6",
                   abs(S[10] - 978.0) < 0.1 and abs(I[10] - 11.4) < 0.1 and abs(R[10] - 10.6) < 0.1))
    day, top = L.peak_of(I)
    checks.append((f"R0 = 2 peak is on day 27 with about 155 infected (got day {day}, {top:.1f})", day == 27 and abs(top - 154.7) < 0.5))
    checks.append(("S falls and R rises every day", bool(np.all(np.diff(S) <= 0) and np.all(np.diff(R) >= 0))))
    checks.append(("about 798.5 people end up in R after 120 days", abs(R[-1] - 798.5) < 0.5))

    checks.append(("R0 = beta / gamma = 2 for beta 0.5, gamma 0.25", abs(L.r0(0.5, 0.25) - 2.0) < 1e-12))
    checks.append(("generations for R0 = 2 are 1, 2, 4, 8, 16, 32", L.generations(2.0) == [1.0, 2.0, 4.0, 8.0, 16.0, 32.0]))
    checks.append(("generations for R0 = 0.5 shrink", L.generations(0.5)[-1] < L.generations(0.5)[0] / 30))

    # R0 above and below 1
    _, I_low, _ = L.run_outbreak(0.125, 0.25)
    checks.append(("R0 = 0.5: infected people fall from day 0", bool(np.all(np.diff(I_low) < 0)) and L.peak_of(I_low)[0] == 0))
    _, I_one, _ = L.run_outbreak(0.25, 0.25)
    checks.append(("R0 = 1: still no growth, peak on day 0", L.peak_of(I_one)[0] == 0))
    # edge case: nobody infected, nothing happens
    S0, I0, R0_ = L.run_outbreak(0.5, 0.25, I0=0)
    checks.append(("with nobody infected nothing changes", bool(np.all(I0 == 0) and np.all(S0 == 1000))))
    # step size check: halving dt changes the peak very little
    _, I_fine, _ = L.run_outbreak(0.5, 0.25, dt=0.05)
    checks.append(("halving the step changes the peak by under 2 people", abs(L.peak_of(I_fine)[1] - top) < 2))
    # assessment answers
    checks.append(("beta 0.3, gamma 0.1 gives R0 = 3 and 27 cases in generation 3", abs(L.r0(0.3, 0.1) - 3) < 1e-12 and L.generations(3.0)[3] == 27.0))
    checks.append(("beta 0.25 makes R0 = 1 and beta 0.2 makes R0 = 0.8 at gamma 0.25",
                   abs(L.r0(0.25, 0.25) - 1) < 1e-12 and abs(L.r0(0.2, 0.25) - 0.8) < 1e-12 and L.peak_of(L.run_outbreak(0.2, 0.25)[1])[0] == 0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
